# Data Profiling: NYC Yellow Taxi 2024

This notebook profiles the cleaned taxi trip dataset for 2024. Before running any analysis or building dashboards, I want to make sure I actually understand what the data looks like — the shape of distributions, where the outliers are, whether the cleaning step worked, and which columns are actually useful.

The source table is `trips_clean`, which was created in `02_clean.sql` by filtering out trips with invalid timestamps, zero fares, and impossible distances from the raw ingested data.

This is not analysis yet. This is just getting familiar with the data before drawing any conclusions from it.

## 0. Setup & Database Connection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sqlalchemy import create_engine, text
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

DB_PASSWORD = 'user'
DB_USER     = 'postgres'
DB_HOST     = 'localhost'
DB_PORT     = '5432'
DB_NAME     = 'NYC Yellow Taxi Trip Report Data 2024'

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

plt.rcParams['figure.figsize']  = (12, 5)
plt.rcParams['axes.spines.top']    = False
plt.rcParams['axes.spines.right']  = False
plt.rcParams['font.family']        = 'DejaVu Sans'
PALETTE = sns.color_palette('Blues_d', 8)

with engine.connect() as conn:
    result = conn.execute(text('SELECT version()'))
    print('Connected to PostgreSQL:', result.scalar()[:40])

## 1. Dataset Overview & Shape

Before profiling anything, we need to understand the basic shape of the data like how many rows, how many columns, and the types of data as well as the time range.

In [ ]:
overview = pd.read_sql_query('''
    SELECT
        COUNT(*)                                                  AS total_rows,
        COUNT(*) FILTER (WHERE suspicious_flag IS NULL)           AS clean_rows,
        COUNT(*) FILTER (WHERE suspicious_flag IS NOT NULL)       AS flagged_rows,
        ROUND(COUNT(*) FILTER (WHERE suspicious_flag IS NOT NULL)
              * 100.0 / COUNT(*), 2)                             AS pct_flagged,
        MIN(tpep_pickup_datetime)                                 AS earliest,
        MAX(tpep_pickup_datetime)                                 AS latest,
        COUNT(DISTINCT DATE_TRUNC(\'month\', tpep_pickup_datetime)) AS months_present
    FROM trips_clean
''', con=engine)

print('=== DATASET OVERVIEW ===')
print(f'Total Rows   : {overview["total_rows"][0]:>15,.0f}')
print(f'Clean Rows   : {overview["clean_rows"][0]:>15,.0f}')
print(f'Flagged Rows : {overview["flagged_rows"][0]:>15,.0f}  ({overview["pct_flagged"][0]}%)')
print(f'Date Range   : {overview["earliest"][0]} → {overview["latest"][0]}')
print(f'Months       : {overview["months_present"][0]} (full year 2024)')

In [ ]:
# Columns schema
schema = pd.read_sql_query('''
    SELECT
        column_name,
        data_type,
        is_nullable
    FROM information_schema.columns
    WHERE table_name = \'trips_clean\'
    ORDER BY ordinal_position
''', con=engine)

print(f'\n=== SCHEMA: {len(schema)} COLUMNS ===')
display(schema)

In [ ]:
# Load a sample for in-memory profiling to avoid loading 39M rows everytime
# Use TABLESAMPLE for a random sample
sample = pd.read_sql_query('''
    SELECT *
    FROM trips_clean
    TABLESAMPLE BERNOULLI(0.5)   -- ~0.5%% of rows, roughly 200k rows
    LIMIT 200000
''', con=engine)

print(f'Sample loaded: {len(sample):,} rows x {len(sample.columns)} columns')
sample.head(3)

## 2. Missing Value Analysis

Missing values can silently distort analysis. We check every column for nulls — both in terms of raw count and percentage — to decide if any columns need special handling before we proceed.

In [ ]:
null_counts = sample.isnull().sum().reset_index()
null_counts.columns = ['column', 'null_count']
null_counts['pct_null'] = (null_counts['null_count'] / len(sample) * 100).round(2)
null_counts = null_counts[null_counts['null_count'] > 0].sort_values('pct_null', ascending=False)

print('=== MISSING VALUE REPORT ===')
if len(null_counts) == 0:
    print('No missing values found in any column.')
else:
    display(null_counts)

In [ ]:
# Visualise missing values if any exist
if len(null_counts) > 0:
    fig, ax = plt.subplots(figsize=(10, max(3, len(null_counts) * 0.5)))
    bars = ax.barh(null_counts['column'], null_counts['pct_null'], color=PALETTE[2])
    ax.set_xlabel('% Missing')
    ax.set_title('Missing Values by Column', fontsize=14, fontweight='bold')
    for bar, pct in zip(bars, null_counts['pct_null']):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                f'{pct:.1f}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig('C:/Users/matth/GitTaxiProject/reports/missing_values.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No missing value chart needed — all columns are complete.')